# Supplemental RT Cross-Test (Training-Distribution Effect)

Cross-evaluates four Regular Transformer (RT) checkpoints — Balanced / ICSD / RRUFF / Augmented — against four matching synthetic test datasets, producing the 4×4 top-1/3/5 matrix backing **`tab:rt_distribution`** (RT training-distribution effect) in the supplement.

Source script: `flash_attn_version/cross_test.py`. The mapping below is a verbatim port of that script's `MODELS` and `TEST_DATASETS` blocks; checkpoint paths are rewritten under `external/checkpoints/` for repo-relative reproduction.

## Environment

```bash
conda activate paper-ai-diffraction-train-eval
pip install -e .
```

GPU strongly recommended — each (model, test-dataset) cell evaluates ~10M synthetic samples. With 4 GPUs and the default `world_size`, rank-0 sees 1/4 of each test set (matching `train.py`'s test behaviour).

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'configs').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'Could not locate repo root from {cwd}')
sys.path.insert(0, str(REPO_ROOT / 'src'))

MODELS_DIR = REPO_ROOT / 'external' / 'checkpoints'
DATA_ROOT  = Path('/scratch/10471/ashetty4428/data')  # external dataset root (Stampede3)

In [ ]:
# Shared RT architecture defaults (cross_test.py); only `depth` differs across rows.
BASE = dict(
    embed_dim    = 256,
    ff_dim       = 128,
    num_heads    = 4,
    mlp_units    = 256,
    dropout      = 0,
    pos_encoding = 'rope',
    num_classes  = 99,
    num_precision = 16,
)

MODELS = [
    {'label': 'Balanced',  'wandb_run_id': 'yv1m76u6', 'depth': 8, 'checkpoint': str(MODELS_DIR / 'xrd_model_yv1m76u6.pth'), **BASE},
    {'label': 'ICSD',      'wandb_run_id': '4hv17ttu', 'depth': 6, 'checkpoint': str(MODELS_DIR / 'xrd_model_4hv17ttu.pth'), **BASE},
    {'label': 'RRUFF',     'wandb_run_id': 'hwixtnv7', 'depth': 6, 'checkpoint': str(MODELS_DIR / 'xrd_model_hwixtnv7.pth'), **BASE},
    {'label': 'Augmented', 'wandb_run_id': 'mq1l94p7', 'depth': 6, 'checkpoint': str(MODELS_DIR / 'xrd_model_mq1l94p7.pth'), **BASE},
]

TEST_DATASETS = [
    {'label': 'Balanced',  'data_path': str(DATA_ROOT / '10Million_pyxtal_RRUFFF-type_repacked.hdf5')},
    {'label': 'ICSD',      'data_path': str(DATA_ROOT / '10mil_pyxtal_rruff-type_icsd_distrib_repacked.hdf5')},
    {'label': 'RRUFF',     'data_path': str(DATA_ROOT / '10mil_pyxtal_rruff-type_rruff_distrib_repacked.hdf5')},
    {'label': 'Augmented', 'data_path': str(DATA_ROOT / 'rruff_aug_10M_trainready_repacked.hdf5')},
]

BATCH_SIZE  = 256
NUM_WORKERS = 8
PREFETCH    = 2
SPEC_LENGTH = 8501  # all four datasets share spec_length=8501 (5–90°)

In [ ]:
import torch
import torch.nn as nn
from collections import OrderedDict

from paper_ai_diffraction.core.rt_model import transformer_model
from paper_ai_diffraction.core.dataset import get_dataloaders_test

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = max(torch.cuda.device_count(), 1)
print(f'Using device: {device}  (world_size={num_gpus})')

def get_dtype(num_precision):
    if num_precision == 16:
        if device.type == 'cuda' and torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16
    return torch.float32

def topk_correct(output, target, k):
    _, pred = output.topk(k, dim=1, largest=True, sorted=True)
    return pred.eq(target.view(-1, 1).expand_as(pred)).any(dim=1).float().sum().item()

def load_model(cfg, spec_length, dtype):
    model = transformer_model(
        spec_length=spec_length,
        num_output=cfg['num_classes'],
        embed_dim=cfg['embed_dim'],
        ff_dim=cfg['ff_dim'],
        depth=cfg['depth'],
        num_heads=cfg['num_heads'],
        mlp_units=cfg['mlp_units'],
        dropout=cfg['dropout'],
        pos_encoding=cfg['pos_encoding'],
    )
    ckpt = torch.load(cfg['checkpoint'], map_location=device)
    state_dict = ckpt.get('model', ckpt)
    clean = OrderedDict((k.replace('module.', ''), v) for k, v in state_dict.items())
    model.load_state_dict(clean, strict=False)
    return model.to(device=device, dtype=dtype).eval()

def evaluate(model, loader, dtype):
    crit = nn.CrossEntropyLoss()
    total = 0; t1 = t3 = t5 = 0; loss = 0.0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs  = inputs.to(device=device, dtype=dtype)
            targets = targets.to(device)
            out = model(inputs)
            loss += crit(out, targets).item()
            total += targets.size(0)
            t1 += topk_correct(out, targets, 1)
            t3 += topk_correct(out, targets, 3)
            t5 += topk_correct(out, targets, 5)
    return dict(top1=100*t1/total, top3=100*t3/total, top5=100*t5/total, total=total)

In [ ]:
# Build all four test loaders once and reuse across models.
print('Loading test dataloaders...')
test_loaders = {}
for ds in TEST_DATASETS:
    _, _, loader, _, _ = get_dataloaders_test(
        ds['data_path'],
        batch_size=BATCH_SIZE,
        world_size=num_gpus,
        num_workers=NUM_WORKERS,
        num_classes=BASE['num_classes'],
        prefetch_factor=PREFETCH,
        start_col=1,
        end_col=SPEC_LENGTH,
        label_mode='categorical',
    )
    test_loaders[ds['label']] = loader
    print(f"  Loaded: {ds['label']:<10} ({ds['data_path']})")

In [ ]:
results = {}

for mcfg in MODELS:
    m_label = mcfg['label']
    dtype   = get_dtype(mcfg['num_precision'])
    print(f"\n{'='*60}\nModel: {m_label}  (run {mcfg['wandb_run_id']}, depth={mcfg['depth']}, dtype={dtype})")

    model = load_model(mcfg, SPEC_LENGTH, dtype)
    for ds in TEST_DATASETS:
        d_label = ds['label']
        m = evaluate(model, test_loaders[d_label], dtype)
        results[(m_label, d_label)] = m
        print(f"  on {d_label:<10}  top-1={m['top1']:.2f}%  top-3={m['top3']:.2f}%  top-5={m['top5']:.2f}%  (n={m['total']})")
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
# Pretty-print 4x4 matrix of top-1/3/5
import pandas as pd

rows = []
for m in MODELS:
    for k in ('top1', 'top3', 'top5'):
        row = {'Train \\ Test': f"{m['label']} ({k})"}
        for ds in TEST_DATASETS:
            r = results.get((m['label'], ds['label']))
            row[ds['label']] = f"{r[k]:.2f}%" if r else 'N/A'
        rows.append(row)

df = pd.DataFrame(rows).set_index('Train \\ Test')
df